# Monte Carlo simulations

In [1]:
import numpy as np
import pandas as pd
import sys
import glob
import plotly.express as px
from pathlib import Path


project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils import load_config
from src.monte_carlo import run_simulation_grid
from src.metrics import summarize_df
from src.figures import frontier_heatmap, estimator_metric_heatmap, frontier_panels, estimator_metric_panels

c:\Users\danil\anaconda3\envs\vcausal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Running MC Sim

In [2]:
config = load_config("baseline")

df = run_simulation_grid(config, save_each=True, n_jobs=8)

alpha_y=1, alpha_d=1, kappa=2: 100%|██████████| 500/500 [01:36<00:00,  5.19it/s]


In [2]:
folder = Path().cwd() / "results" / "raw"

files = list(folder.glob("*.parquet"))


all_results = [pd.read_parquet(f) for f in files]
df = pd.concat(all_results, ignore_index=True)

In [10]:
df

,alpha_y,alpha_d,kappa,seed,replication,tau_true,ols_tau_hat,ols_se,ols_ci_lower,ols_ci_upper,dml_tau_hat,dml_se,dml_ci_lower,dml_ci_upper
0,0.25,0.25,0.5,42,0,2.5,2.371643,0.095408,2.184643,2.558644,2.449772,0.102224,2.249414,2.650131
1,0.25,0.25,0.5,43,1,2.5,2.608884,0.094026,2.424594,2.793174,2.669094,0.106497,2.460359,2.877828
2,0.25,0.25,0.5,44,2,2.5,2.651157,0.097307,2.460436,2.841879,2.608289,0.107842,2.396919,2.819660
3,0.25,0.25,0.5,45,3,2.5,2.529472,0.097213,2.338935,2.720009,2.544056,0.105458,2.337358,2.750754
4,0.25,0.25,0.5,46,4,2.5,2.555124,0.097280,2.364455,2.745794,2.564021,0.104488,2.359225,2.768818
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37495,1.00,1.00,2.0,537,495,2.5,2.046252,0.198710,1.656780,2.435723,2.054218,0.133022,1.793495,2.314940
37496,1.00,1.00,2.0,538,496,2.5,2.487868,0.229676,2.037703,2.938032,2.194140,0.137284,1.925064,2.463217
37497,1.00,1.00,2.0,539,497,2.5,2.566798,0.151712,2.269442,2.864154,2.475612,0.139979,2.201254,2.749970
37498,1.00,1.00,2.0,540,498,2.5,2.713088,0.167299,2.385182,3.040994,2.472255,0.125018,2.227221,2.717289


In [3]:
summary_df = summarize_df(df)
summary_df.head()

,estimator,bias,sd,rmse,coverage,avg_ci_length,alpha_y,alpha_d,kappa
0,OLS,-0.005495,0.097503,0.097560,0.936,0.367338,0.0,0.0,0.5
1,DML,0.009291,0.114778,0.115039,0.944,0.426882,0.0,0.0,0.5
2,OLS,-0.003479,0.104804,0.104757,0.938,0.391400,0.0,0.0,1.0
3,DML,-0.047974,0.124172,0.133001,0.908,0.453424,0.0,0.0,1.0
4,OLS,0.001120,0.125216,0.125095,0.926,0.448260,0.0,0.0,2.0


## Frontiers


In [9]:
fig = frontier_heatmap(summary_df, kappa=0.5)
fig.show()

In [5]:
fig = estimator_metric_heatmap(summary_df, metric="bias", estimator="OLS", kappa=1)
fig.show()


In [6]:
fig = estimator_metric_heatmap(summary_df, metric="bias", estimator="DML", kappa=1)
fig.show()

In [13]:
fig = estimator_metric_heatmap(summary_df, metric="sd", estimator="DML", kappa=1)
fig.show()

In [12]:
fig = estimator_metric_heatmap(summary_df, metric="sd", estimator="OLS", kappa=1)
fig.show()

In [15]:
fig = estimator_metric_heatmap(summary_df, metric="coverage", estimator="DML", kappa=1)
fig.show()

In [14]:
fig = estimator_metric_heatmap(summary_df, metric="coverage", estimator="OLS", kappa=1)
fig.show()

In [9]:
fig = frontier_panels(summary_df, kappas=[0.5, 1, 2.0])
fig.show()

In [4]:
fig = estimator_metric_panels(summary_df, metric="bias", estimator="OLS", kappas=[ 0.5, 1.0, 2])
fig.show()

In [5]:
fig = estimator_metric_panels(summary_df, metric="bias", estimator="DML", kappas=[ 0.5, 1.0, 2])
fig.show()

In [6]:
fig = estimator_metric_panels(summary_df, metric="sd", estimator="OLS", kappas=[ 0.5, 1.0, 2])
fig.show()

In [8]:
fig = estimator_metric_panels(summary_df, metric="sd", estimator="DML", kappas=[ 0.5, 1.0, 2])
fig.show()

## Supporting figures

In [11]:
import numpy as np
import plotly.graph_objects as go

def theoretical_3d_surface(
    alpha_y_grid=np.linspace(0, 1, 60),
    kappa_grid=np.linspace(0.5, 2.0, 60),
    alpha_d=0.5,
    a=1.2,
    b=0.55,
    c=0.25,
    offset=0.35
):
    AY, K = np.meshgrid(alpha_y_grid, kappa_grid, indexing="xy")
    S = a * AY + c * alpha_d - b * (K ** 2) - offset

    fig = go.Figure(
        data=[
            go.Surface(
                x=AY,
                y=K,
                z=S,
                colorscale="RdBu",
                colorbar=dict(title="Stylized score")
            )
        ]
    )

    fig.update_layout(
        title=f"Stylized DML advantage surface (alpha_d = {alpha_d})",
        scene=dict(
            xaxis_title="alpha_y",
            yaxis_title="kappa",
            zaxis_title="DML advantage score"
        ),
        template="plotly_white"
    )

    return fig

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def theoretical_dml_frontier_by_alpha_d(
    alpha_d_values=(0.0, 0.5, 1.0),
    alpha_y_grid=np.linspace(0, 1, 101),
    kappa_grid=np.linspace(0.5, 2.0, 101),
    a=1.2,   # benefit from outcome nonlinearity
    b=0.55,  # overlap penalty
    c=0.25,  # treatment learnability benefit
    offset=0.35
):
    """
    Stylized score:
        S(alpha_y, kappa, alpha_d)
            = a * alpha_y + c * alpha_d - b * kappa**2 - offset

    Interpretation:
        S > 0 => DML better
        S < 0 => OLS better
    """
    fig = make_subplots(
        rows=1,
        cols=len(alpha_d_values),
        subplot_titles=[f"Kappa = {ad}" for ad in alpha_d_values],
        shared_yaxes=True
    )

    for i, alpha_d in enumerate(alpha_d_values, start=1):
        AY, K = np.meshgrid(alpha_y_grid, kappa_grid, indexing="xy")
        S = a * AY + c * alpha_d - b * (K ** 2) - offset

        fig.add_trace(
            go.Contour(
                x=alpha_y_grid,
                y=kappa_grid,
                z=S,
                colorscale="RdBu",
                zmin=-1.5,
                zmax=1.0,
                showscale=(i == len(alpha_d_values)),
                colorbar=dict(title="Stylized score"),
                contours=dict(
                    start=-1.5,
                    end=1.0,
                    size=0.1
                ),
                hovertemplate=(
                    "alpha_y=%{x:.2f}<br>"
                    "kappa=%{y:.2f}<br>"
                    f"alpha_d={alpha_d:.2f}<br>"
                    "score=%{z:.2f}<extra></extra>"
                )
            ),
            row=1, col=i
        )

        fig.add_trace(
            go.Contour(
                x=alpha_y_grid,
                y=kappa_grid,
                z=S,
                contours=dict(
                    start=0,
                    end=0,
                    coloring="lines"
                ),
                line=dict(width=3),
                showscale=False,
                hoverinfo="skip"
            ),
            row=1, col=i
        )

    fig.update_xaxes(title_text="Outcome nonlinearity (alpha_y)")
    fig.update_yaxes(title_text="Overlap deterioration (kappa)")

    fig.update_layout(
        title="Stylized DML-vs-OLS frontier across treatment complexity",
        template="plotly_white"
    )

    return fig

In [15]:
theoretical_dml_frontier_by_alpha_d()

In [16]:
import numpy as np
import plotly.graph_objects as go

def theoretical_dml_frontier(
    alpha_y_grid=np.linspace(0, 1, 101),
    kappa_grid=np.linspace(0.5, 2.0, 101),
    a=1.2,   # benefit from outcome nonlinearity
    b=0.55,  # cost from poor overlap
    offset=0.35
):
    """
    Stylized score:
        S(alpha_y, kappa) = a * alpha_y - b * kappa**2 - offset

    Interpretation:
        S > 0  => DML outperforms OLS
        S < 0  => OLS outperforms DML

    This is a theoretical intuition plot, not estimated from simulation data.
    """
    AY, K = np.meshgrid(alpha_y_grid, kappa_grid, indexing="xy")
    S = a * AY - b * (K ** 2) - offset

    fig = go.Figure()

    fig.add_trace(go.Contour(
        x=alpha_y_grid,
        y=kappa_grid,
        z=S,
        colorscale="RdBu",
        contours=dict(
            start=S.min(),
            end=S.max(),
            size=(S.max() - S.min()) / 20
        ),
        colorbar=dict(title="Stylized score"),
        hovertemplate=(
            "alpha_y=%{x:.2f}<br>"
            "kappa=%{y:.2f}<br>"
            "score=%{z:.2f}<extra></extra>"
        )
    ))

    # Boundary S = 0
    fig.add_trace(go.Contour(
        x=alpha_y_grid,
        y=kappa_grid,
        z=S,
        contours=dict(
            start=0,
            end=0,
            coloring="lines"
        ),
        line=dict(width=3),
        showscale=False,
        hoverinfo="skip"
    ))

    fig.update_layout(
        title="Stylized frontier: where DML outperforms OLS",
        xaxis_title="Outcome nonlinearity (alpha_y)",
        yaxis_title="Overlap deterioration (kappa)",
        template="plotly_white"
    )

    return fig

In [17]:
theoretical_dml_frontier()

In [18]:
import pandas as pd
import numpy as np

def rmse_diff_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute RMSE for OLS and DML and return RMSE difference:
        rmse_diff = RMSE_OLS - RMSE_DML

    Grouped by (alpha_y, alpha_d, kappa)
    """

    data = df.copy()

    # Squared errors
    data["ols_se_sq"] = (data["ols_tau_hat"] - data["tau_true"]) ** 2
    data["dml_se_sq"] = (data["dml_tau_hat"] - data["tau_true"]) ** 2

    # Group and compute RMSE
    grouped = (
        data.groupby(["alpha_y", "alpha_d", "kappa"])
        .agg(
            ols_rmse=("ols_se_sq", lambda x: np.sqrt(np.mean(x))),
            dml_rmse=("dml_se_sq", lambda x: np.sqrt(np.mean(x))),
        )
        .reset_index()
    )

    # RMSE difference
    grouped["rmse_diff"] = grouped["ols_rmse"] - grouped["dml_rmse"]

    return grouped

In [20]:
rmse_dif = rmse_diff_table(df)

In [21]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def empirical_rmse_contours(rmse_df: pd.DataFrame):
    """
    Data-driven contour plots of RMSE difference:
        rmse_diff = RMSE_OLS - RMSE_DML

    Positive values => DML better
    Negative values => OLS better

    Axes:
        x = alpha_d
        y = alpha_y
    Panels:
        one per kappa
    """
    kappas = sorted(rmse_df["kappa"].unique())
    n = len(kappas)

    fig = make_subplots(
        rows=1,
        cols=n,
        subplot_titles=[f"kappa = {k}" for k in kappas],
        shared_yaxes=True
    )

    zmin = rmse_df["rmse_diff"].min()
    zmax = rmse_df["rmse_diff"].max()
    absmax = max(abs(zmin), abs(zmax))

    for i, kappa in enumerate(kappas, start=1):
        sub = rmse_df[rmse_df["kappa"] == kappa].copy()

        pivot = (
            sub.pivot(index="alpha_y", columns="alpha_d", values="rmse_diff")
            .sort_index()
            .sort_index(axis=1)
        )

        x = pivot.columns.to_numpy()
        y = pivot.index.to_numpy()
        z = pivot.to_numpy()

        # Filled contour
        fig.add_trace(
            go.Contour(
                x=x,
                y=y,
                z=z,
                colorscale="RdBu",
                zmid=0,
                zmin=-absmax,
                zmax=absmax,
                contours=dict(
                    start=-absmax,
                    end=absmax,
                    size=absmax / 10 if absmax > 0 else 0.01
                ),
                colorbar=dict(title="RMSE diff") if i == n else None,
                showscale=(i == n),
                hovertemplate=(
                    "alpha_d=%{x:.2f}<br>"
                    "alpha_y=%{y:.2f}<br>"
                    "rmse_diff=%{z:.4f}<extra></extra>"
                )
            ),
            row=1, col=i
        )

        # Zero contour frontier
        fig.add_trace(
            go.Contour(
                x=x,
                y=y,
                z=z,
                contours=dict(
                    start=0,
                    end=0,
                    coloring="lines"
                ),
                line=dict(width=3, color="black"),
                showscale=False,
                hoverinfo="skip"
            ),
            row=1, col=i
        )

    fig.update_xaxes(title_text="Treatment complexity (alpha_d)")
    fig.update_yaxes(title_text="Outcome nonlinearity (alpha_y)")

    fig.update_layout(
        title="Empirical frontier: regions where DML outperforms OLS",
        template="plotly_white"
    )

    return fig

In [22]:
empirical_rmse_contours(rmse_dif)

In [23]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def combined_theory_empirical_frontier(
    rmse_df: pd.DataFrame,
    kappa_values=(0.5, 1.0, 2.0),
    alpha_y_grid=np.linspace(0, 1, 101),
    alpha_d_grid=np.linspace(0, 1, 101),
    a=1.2,   # benefit from outcome nonlinearity
    b=0.25,  # smaller benefit from treatment learnability
    c=0.55,  # overlap penalty
    d=0.45   # offset
):
    """
    Create a 2xK panel figure:
      - top row: theoretical contour plots
      - bottom row: empirical contour plots based on rmse_diff

    Theoretical score:
        S(alpha_y, alpha_d; kappa) = a*alpha_y + b*alpha_d - c*kappa^2 - d

    Interpretation:
        S > 0  => DML better
        S < 0  => OLS better
    """

    kappa_values = list(kappa_values)
    ncols = len(kappa_values)

    fig = make_subplots(
        rows=2,
        cols=ncols,
        shared_xaxes=False,
        shared_yaxes=False,
        vertical_spacing=0.12,
        horizontal_spacing=0.06,
        subplot_titles=(
            [f"Theory: kappa = {k}" for k in kappa_values] +
            [f"Empirical: kappa = {k}" for k in kappa_values]
        )
    )

    # ---------- Top row: theoretical ----------
    AY, AD = np.meshgrid(alpha_y_grid, alpha_d_grid, indexing="ij")

    theory_scores = {}
    theory_absmax = 0.0

    for kappa in kappa_values:
        S = a * AY + b * AD - c * (kappa ** 2) - d
        theory_scores[kappa] = S
        theory_absmax = max(theory_absmax, np.max(np.abs(S)))

    for col, kappa in enumerate(kappa_values, start=1):
        S = theory_scores[kappa]

        fig.add_trace(
            go.Contour(
                x=alpha_d_grid,
                y=alpha_y_grid,
                z=S,
                colorscale="RdBu",
                zmid=0,
                zmin=-theory_absmax,
                zmax=theory_absmax,
                showscale=(col == ncols),
                colorbar=dict(
                    title="Theory score",
                    x=1.02,
                    y=0.80,
                    len=0.35
                ) if col == ncols else None,
                contours=dict(
                    start=-theory_absmax,
                    end=theory_absmax,
                    size=theory_absmax / 12 if theory_absmax > 0 else 0.1
                ),
                line=dict(width=0.6),
                hovertemplate=(
                    "alpha_d=%{x:.2f}<br>"
                    "alpha_y=%{y:.2f}<br>"
                    f"kappa={kappa:.1f}<br>"
                    "score=%{z:.3f}<extra></extra>"
                )
            ),
            row=1, col=col
        )

        # Zero contour
        fig.add_trace(
            go.Contour(
                x=alpha_d_grid,
                y=alpha_y_grid,
                z=S,
                contours=dict(
                    start=0,
                    end=0,
                    coloring="lines"
                ),
                line=dict(width=4, color="gold"),
                showscale=False,
                hoverinfo="skip"
            ),
            row=1, col=col
        )

    # ---------- Bottom row: empirical ----------
    emp_absmax = max(abs(rmse_df["rmse_diff"].min()), abs(rmse_df["rmse_diff"].max()))

    for col, kappa in enumerate(kappa_values, start=1):
        sub = rmse_df[rmse_df["kappa"] == kappa].copy()

        pivot = (
            sub.pivot(index="alpha_y", columns="alpha_d", values="rmse_diff")
            .sort_index()
            .sort_index(axis=1)
        )

        x = pivot.columns.to_numpy()
        y = pivot.index.to_numpy()
        z = pivot.to_numpy()

        fig.add_trace(
            go.Contour(
                x=x,
                y=y,
                z=z,
                colorscale="RdBu",
                zmid=0,
                zmin=-emp_absmax,
                zmax=emp_absmax,
                showscale=(col == ncols),
                colorbar=dict(
                    title="RMSE diff",
                    x=1.02,
                    y=0.20,
                    len=0.35
                ) if col == ncols else None,
                contours=dict(
                    start=-emp_absmax,
                    end=emp_absmax,
                    size=emp_absmax / 12 if emp_absmax > 0 else 0.01
                ),
                line=dict(width=0.6),
                hovertemplate=(
                    "alpha_d=%{x:.2f}<br>"
                    "alpha_y=%{y:.2f}<br>"
                    f"kappa={kappa:.1f}<br>"
                    "rmse_diff=%{z:.4f}<extra></extra>"
                )
            ),
            row=2, col=col
        )

        # Zero contour
        if np.nanmin(z) <= 0 <= np.nanmax(z):
            fig.add_trace(
                go.Contour(
                    x=x,
                    y=y,
                    z=z,
                    contours=dict(
                        start=0,
                        end=0,
                        coloring="lines"
                    ),
                    line=dict(width=4, color="gold"),
                    showscale=False,
                    hoverinfo="skip"
                ),
                row=2, col=col
            )

    # ---------- Axes ----------
    for col in range(1, ncols + 1):
        fig.update_xaxes(title_text="Treatment complexity (alpha_d)", row=1, col=col)
        fig.update_xaxes(title_text="Treatment complexity (alpha_d)", row=2, col=col)
        fig.update_yaxes(title_text="Outcome nonlinearity (alpha_y)", row=1, col=col)
        fig.update_yaxes(title_text="Outcome nonlinearity (alpha_y)", row=2, col=col)

    fig.update_layout(
        title="Theoretical and empirical frontiers for DML vs OLS",
        template="plotly_white",
        width=1600,
        height=950,
        margin=dict(t=90, l=60, r=120, b=60)
    )

    return fig

In [25]:
fig = combined_theory_empirical_frontier(rmse_dif)
fig.show()